<!--
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0
-->
# AVF Demo (Colab)
Este notebook demonstra a criação de um arquivo .avf, replay com snapshots e integração com LLM.

In [ ]:
# Instalação opcional de dependências
!pip -q install transformers torch || true

In [ ]:
import os
from pathlib import Path
from runtime.avf_core import CognitiveCore, PersistentIdentity
from runtime.avf_io import AVFAppendWriter, AVFStreamReader
from runtime.llm_adapter import MockLLM
from runtime.snapshot import write_snapshot

base_path = Path('/content' if os.path.exists('/content') else '/mnt/data')
base_path.mkdir(parents=True, exist_ok=True)
avf_path = base_path / 'demo.avf'
writer = AVFAppendWriter(str(avf_path))
core = CognitiveCore()
core.identity = PersistentIdentity('colab-demo', {'tone': 'curious'})
core.persist_identity(writer)

for idx in range(10):
    writer.append_chunk('EVENT', {'type': 'concept', 'value': f'c{idx}'})
    if idx == 5:
        write_snapshot(writer, core.state)

print('AVF criado em', avf_path)

In [ ]:
# Replay e leitura streaming
reader = AVFStreamReader(str(avf_path))
for chunk in reader.stream():
    print(chunk['type'], chunk['payload'])

In [ ]:
# Simular restart e recuperar identidade
core2 = CognitiveCore()
core2.reconstruct_from_avf(str(avf_path))
print('Identity:', core2.identity)

In [ ]:
# Integração com LLM (Mock)
llm = MockLLM(prefix='demo')
response = core2.think_with_llm(llm, writer, 'Olá, AVF!')
print(response)

In [ ]:
# Métricas simples de write e replay
import json
import time

start_write = time.time()
for idx in range(50):
    writer.append_chunk('EVENT', {'type': 'metric', 'value': idx})
write_time = time.time() - start_write

start_replay = time.time()
core3 = CognitiveCore()
core3.reconstruct_from_avf(str(avf_path))
replay_time = time.time() - start_replay

report = {'write_time': write_time, 'replay_time': replay_time}
report_path = base_path / 'avf_test_report.json'
report_path.write_text(json.dumps(report, indent=2))
print('Report saved:', report_path)